<a href="https://colab.research.google.com/github/ghinaiyariken/Fly_Rank_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghinaiyariken/Fly_Rank_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Two signals checked first**

1. **Staleness:** `days_since_last_update`, bucketed as `0-90`, `91-180`, `181+`. This is directly behind FlyRank's refresh/staleness flags.
2. **Search visibility:** `impressions_90d`, bucketed as `1-99`, `100-499`, `500-2,999`, `3,000-29,999`, `30,000+`. This is the volume signal behind quick-win logic.

I use `trend_direction` only to audit whether these observed signals are directionally useful. It is **not** an input to the score.

**Verdicts:**
- **Staleness — MIXED:** 91–180 days has a higher observed decline rate than 0–90, but the 181+ bucket is very small and is not higher, so I will not claim a monotonic effect.
- **Visibility — CONFIRMED:** the 100–29,999 impression buckets have higher observed decline rates than the 1–99 bucket. The 30,000+ bucket falls again, so I use visibility as an evidence gate rather than assuming “more impressions = more decline.”

**Rule:** review a page first when it is both **stale (≥180 days since update)** and **visible (≥500 impressions in 90 days)**. Score = `1` when both conditions are true, otherwise `0`. The rule uses only information available in the current 90-day snapshot; no future-window or label-derived field enters the score.

**Reason code:** `stale_visible_page` when score = 1; `not_priority` otherwise.

**Action:** `review_refresh` when score = 1; `monitor` otherwise. The action is a human-review recommendation, not a claim that refreshing will cause recovery.


In [2]:
# Signal checks: visible bucket tables with n and observed decline rate.
import pandas as pd
import numpy as np

DATA_PATH = "content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"], bins=[-1, 90, 180, np.inf],
    labels=["0-90", "91-180", "181+"],
)
df["visibility_bucket"] = pd.cut(
    df["impressions_90d"], bins=[0, 99, 499, 2999, 29999, np.inf],
    labels=["1-99", "100-499", "500-2,999", "3,000-29,999", "30,000+"],
)

def bucket_table(col):
    return (df.groupby(col, observed=False)
              .agg(n=("content_id", "size"),
                   decline_rate=("trend_direction", lambda s: (s == "down").mean()))
              .reset_index())

print("Staleness signal")
display(bucket_table("staleness_bucket"))
print("Visibility signal")
display(bucket_table("visibility_bucket"))
print()
print("Note: decline_rate is an audit outcome only. It is never used in the score.")


Staleness signal


,staleness_bucket,n,decline_rate
0,0-90,20655,0.512031
1,91-180,9171,0.611057
2,181+,174,0.471264


Visibility signal


,visibility_bucket,n,decline_rate
0,1-99,7994,0.389042
1,100-499,5280,0.604356
2,"500-2,999",8443,0.620632
3,"3,000-29,999",7205,0.586121
4,"30,000+",1078,0.461967



Note: decline_rate is an audit outcome only. It is never used in the score.


## 2. Build the ranked queue (writes the CSV)

The score is intentionally simple and frozen: **1 = stale + visible, 0 = otherwise**. It is transparent, has one reason code, and produces a ranked review queue. Ties are broken by impressions so higher-volume pages appear first within the same score.


In [3]:
# Encode ONE transparent rule, add one reason code and one action label, then write the queue.
# No trend/label field is used in the score.

stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500

df["score"] = (stale & visible).astype(int)
df["reason_code"] = np.where(df["score"].eq(1), "stale_visible_page", "not_priority")
df["action"] = np.where(df["score"].eq(1), "review_refresh", "monitor")

df["rank"] = (
    df.sort_values(["score", "impressions_90d"], ascending=[False, False])
      .reset_index()
      .index + 1
)
# The rank above follows sorted order; re-map it to original rows.
sorted_idx = df.sort_values(["score", "impressions_90d"], ascending=[False, False]).index
df.loc[sorted_idx, "rank"] = np.arange(1, len(df) + 1)

queue_cols = ["rank", "content_id", "score", "reason_code", "action",
              "days_since_last_update", "impressions_90d", "sessions_90d"]
queue = df.sort_values("rank")[queue_cols].copy()

out_path = "work/outputs/baseline_action_score.csv"
import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv(out_path, index=False)

print(f"Rows ranked: {len(queue):,}")
print(f"Flagged for review: {(queue['score'] == 1).sum():,}")
print(f"CSV written: {out_path}")
print("\nTop 10:")
display(queue.head(10))


Rows ranked: 30,000
Flagged for review: 17
CSV written: work/outputs/baseline_action_score.csv

Top 10:


,rank,content_id,score,reason_code,action,days_since_last_update,impressions_90d,sessions_90d
16751,1,content_cf56e2e2e282,1,stale_visible_page,review_refresh,194,61678,119
16514,2,content_7368877ea310,1,stale_visible_page,review_refresh,194,59472,82
7021,3,content_1bfaa38ff26c,1,stale_visible_page,review_refresh,194,25715,80
21268,4,content_0a91db491d14,1,stale_visible_page,review_refresh,193,13299,78
11489,5,content_5feee3994adb,1,stale_visible_page,review_refresh,194,7812,5
12045,6,content_c2d929d83eaa,1,stale_visible_page,review_refresh,193,7558,25
698,7,content_b16bd7307b39,1,stale_visible_page,review_refresh,194,4590,4
5327,8,content_fe16a55cd13d,1,stale_visible_page,review_refresh,194,4556,42
26810,9,content_ecb6215e79fd,1,stale_visible_page,review_refresh,194,4429,12
20837,10,content_928af3e22c80,1,stale_visible_page,review_refresh,193,1697,3


## 3. Top-10 review

The top 10 are the highest-scoring pages under the frozen rule. For each row, the review note records the action, why it was selected, and what could make the recommendation wrong. The rule is a prioritization aid, not proof that a refresh will improve performance.


In [4]:
# Review each of the top 10 rows. IDs are pseudonymous; no client names, URLs, or private queries are shown.
top10 = queue.head(10).copy()

def confidence(row):
    if row["impressions_90d"] >= 3000 and row["sessions_90d"] >= 30:
        return "higher: strong observed volume and sessions"
    if row["impressions_90d"] >= 500 and row["sessions_90d"] >= 10:
        return "medium: enough observed volume, but review context is still limited"
    return "lower: visibility passes the gate but session evidence is thin"

def wrong_if(row):
    if row["sessions_90d"] < 10:
        return "wrong if low session volume is mostly noise or the page is not worth changing"
    return "wrong if the decline is caused by seasonality, SERP/query changes, or a non-content issue"

review = pd.DataFrame({
    "rank": top10["rank"],
    "action": top10["action"],
    "reason": top10["reason_code"],
    "confidence_note": top10.apply(confidence, axis=1),
    "what_would_make_it_wrong": top10.apply(wrong_if, axis=1),
})
display(review.to_string(index=False))

# Precision@10 is shown only as a diagnostic against the observed starter label.
# The label is NOT used by the scoring rule.
precision_at_10 = (top10["content_id"].map(
    df.set_index("content_id")["trend_direction"]
).eq("down")).mean()
base_rate = (df["trend_direction"] == "down").mean()
print(f"\nDiagnostic precision@10 vs observed decline label: {precision_at_10:.3f}")
print(f"Overall observed decline base rate: {base_rate:.3f}")


' rank         action             reason                                                     confidence_note                                                                  what_would_make_it_wrong\n    1 review_refresh stale_visible_page                         higher: strong observed volume and sessions wrong if the decline is caused by seasonality, SERP/query changes, or a non-content issue\n    2 review_refresh stale_visible_page                         higher: strong observed volume and sessions wrong if the decline is caused by seasonality, SERP/query changes, or a non-content issue\n    3 review_refresh stale_visible_page                         higher: strong observed volume and sessions wrong if the decline is caused by seasonality, SERP/query changes, or a non-content issue\n    4 review_refresh stale_visible_page                         higher: strong observed volume and sessions wrong if the decline is caused by seasonality, SERP/query changes, or a non-content issue\n    


Diagnostic precision@10 vs observed decline label: 1.000
Overall observed decline base rate: 0.542


## 4. Weak picks + leakage check

**Weak-pick review:** the top 10 are all rule-positive because the rule has a strict stale+visible gate. That does not make them certain refresh wins. The weaker cases are the rows with low `sessions_90d`: they have search visibility, but less direct behavior evidence, so they deserve more skepticism before action.

**Leakage check:** the score uses only `days_since_last_update` and `impressions_90d`. It does not use `trend_direction`, `trend_pct`, `is_declining_label`, future windows, model probabilities, or FlyRank product flags. The observed decline label is used only after ranking to calculate a diagnostic precision@10; it never affects ranking.


In [5]:
# Explicit leakage audit: verify forbidden label-derived/future fields are absent from the score inputs.
score_inputs = {"days_since_last_update", "impressions_90d"}
forbidden = {"trend_direction", "trend_pct", "is_declining_label", "declined_next_month", "future_impression_change_pct", "model_probability", "priority_score", "health_score", "action_type"}

print("Score inputs:", sorted(score_inputs))
print("Forbidden fields checked:", sorted(forbidden))
assert score_inputs.isdisjoint(forbidden)

# Sanity check that the output exists and has the required columns.
required = {"rank", "content_id", "score", "reason_code", "action"}
assert required.issubset(queue.columns)
assert len(queue) == len(df)
print("PASS: no future-window or label-derived field is used by the baseline score.")
print("PASS: ranked queue has one row per starter content item.")


Score inputs: ['days_since_last_update', 'impressions_90d']
Forbidden fields checked: ['action_type', 'declined_next_month', 'future_impression_change_pct', 'health_score', 'is_declining_label', 'model_probability', 'priority_score', 'trend_direction', 'trend_pct']
PASS: no future-window or label-derived field is used by the baseline score.
PASS: ranked queue has one row per starter content item.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] The two signal checks show bucket labels and `n`; at least one signal is directly behind FlyRank's session refresh/quick-win flags
- [x] One transparent rule produces a score, one reason code, and an action label
- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`
- [x] Top-10 review includes action, reason, confidence, and what could make each recommendation wrong
- [x] No future-window or label-derived input is used in the score
- [ ] Commit `work/notebooks/w04_baseline_score.ipynb` to my repo
- [ ] Submit the repo URL on the ML-07 card
